# Pipeline ETL - Nettoyage et Transformation des Données

## Objectifs de ce notebook :
1. Charger les données du jour 1
2. Analyser la qualité des données
3. Nettoyer les valeurs manquantes
4. Corriger les types de données
5. Créer de nouvelles variables (feature engineering)
6. Exporter les données propres

**Donnée d'entrée** : `dataco_subset.csv` (créé au Jour 1)
**Donnée de sortie** : `dataco_clean.csv` (pour Power BI et le modèle ML)

In [2]:
# Importation des bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print(" Bibliothèques importées")

 Bibliothèques importées


In [3]:
# Chargement des données du jour 1
df = pd.read_csv('../data/processed/dataco_subset.csv')

print(f" Données chargées : {df.shape[0]:,} lignes, {df.shape[1]} colonnes")
print(f" Colonnes : {df.columns.tolist()}")

 Données chargées : 180,519 lignes, 15 colonnes
 Colonnes : ['Order Id', 'Customer City', 'Order Country', 'order date (DateOrders)', 'shipping date (DateOrders)', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Late_delivery_risk', 'Delivery Status', 'Order Item Quantity', 'Sales per customer', 'Benefit per order', 'Category Name', 'Shipping Mode', 'Customer Segment']


## 1. Analyse de la qualité des données

Avant de nettoyer, il faut comprendre ce qui ne va pas :
- Valeurs manquantes
- Types de données incorrects
- Doublons
- Valeurs aberrantes

In [4]:
# 1.1. Vérification des valeurs manquantes
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]
missing_values = missing_values.sort_values(ascending=False)

print("Colonnes avec des valeurs manquantes :")
print(missing_values)

if not missing_values.empty:
    # Visualisation
    plt.figure(figsize=(10, 5))
    missing_values.plot(kind='bar', color='coral')
    plt.title('Valeurs manquantes par colonne', fontsize=14)
    plt.xlabel('Colonnes')
    plt.ylabel('Nombre de valeurs manquantes')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # Pourcentage de valeurs manquantes
    for col, val in missing_values.items():
        pct = (val / len(df)) * 100
        print(f"  - {col}: {val:,} ({pct:.2f}%)")
else:
    print("Aucune valeur manquante détectée dans le dataset.")

Colonnes avec des valeurs manquantes :
Series([], dtype: int64)
Aucune valeur manquante détectée dans le dataset.


In [5]:
# 1.2. Vérification des types de données
print("Types de données actuels :")
print(df.dtypes)

Types de données actuels :
Order Id                           int64
Customer City                     object
Order Country                     object
order date (DateOrders)           object
shipping date (DateOrders)        object
Days for shipping (real)           int64
Days for shipment (scheduled)      int64
Late_delivery_risk                 int64
Delivery Status                   object
Order Item Quantity                int64
Sales per customer               float64
Benefit per order                float64
Category Name                     object
Shipping Mode                     object
Customer Segment                  object
dtype: object


In [6]:
# 1.3. Vérification des doublons
doublons = df.duplicated().sum()
print(f"Nombre de lignes en double : {doublons:,}")

if doublons > 0:
    print(f"{doublons} doublons détectés. Ils seront supprimés.")
else:
    print("Aucun doublon détecté.")

Nombre de lignes en double : 0
Aucun doublon détecté.


## 2. Nettoyage des valeurs manquantes
## Même ci le bloc précédent montre qu'on a aucune valeur manquante mais on va faire cette étape comme étape de sécurité pour des futurs jeux de données.
### Stratégies de nettoyage :

| Colonne | Problème | Solution |
|---------|----------|----------|
| `Customer City` | Valeurs manquantes | Remplacer par 'Unknown' |
| `Order Country` | Valeurs manquantes | Remplacer par 'Unknown' |
| `Delivery Status` | Valeurs manquantes | Remplacer par 'Unknown' |
| `Customer Segment` | Valeurs manquantes | Remplacer par 'Unknown' |
| `Category Name` | Valeurs manquantes | Remplacer par 'Unknown' |
| `Shipping Mode` | Valeurs manquantes | Remplacer par 'Unknown' |

In [7]:
# 2.1. Remplacer les valeurs manquantes dans les colonnes catégorielles
colonnes_categoriques = [
    'Customer City',
    'Order Country', 
    'Delivery Status',
    'Customer Segment',
    'Category Name',
    'Shipping Mode'
]

for col in colonnes_categoriques:
    if col in df.columns:
        nb_null = df[col].isnull().sum()
        if nb_null > 0:
            df[col] = df[col].fillna('Unknown')
            print(f" {col}: {nb_null} valeurs manquantes remplacées par 'Unknown'")

In [8]:
# 2.2. Vérification après correction
missing_after = df.isnull().sum()
missing_after = missing_after[missing_after > 0]

if len(missing_after) == 0:
    print(" Plus aucune valeur manquante !")
else:
    print(f" Il reste {len(missing_after)} colonnes avec des valeurs manquantes :")
    print(missing_after)

 Plus aucune valeur manquante !


## 3. Correction des types de données

Les colonnes de dates doivent être converties en format datetime pour pouvoir les manipuler facilement.

In [9]:
# 3.1. Convertir les colonnes de date
colonnes_dates = [
    'order date (DateOrders)',
    'shipping date (DateOrders)'
]

for col in colonnes_dates:
    if col in df.columns:
        try:
            df[col] = pd.to_datetime(df[col])
            print(f" {col} convertie en datetime")
        except Exception as e:
            print(f" Erreur pour {col}: {e}")

 order date (DateOrders) convertie en datetime
 shipping date (DateOrders) convertie en datetime


In [10]:
# 3.2. Vérifier les types après conversion
print(" Types de données après correction :")
print(df.dtypes)

 Types de données après correction :
Order Id                                  int64
Customer City                            object
Order Country                            object
order date (DateOrders)          datetime64[ns]
shipping date (DateOrders)       datetime64[ns]
Days for shipping (real)                  int64
Days for shipment (scheduled)             int64
Late_delivery_risk                        int64
Delivery Status                          object
Order Item Quantity                       int64
Sales per customer                      float64
Benefit per order                       float64
Category Name                            object
Shipping Mode                            object
Customer Segment                         object
dtype: object


## 4. Feature Engineering

Création de nouvelles variables (features) qui seront utiles pour :
- L'analyse dans Power BI
- Le modèle de prédiction

### Nouvelles colonnes à créer :

| Nouvelle colonne | Description | Utilité |
|------------------|-------------|---------|
| `Delay_Flag` | 1 si retard, 0 sinon | Variable cible pour ML |
| `Delay_Days` | Nombre de jours de retard | Analyse des retards |
| `Is_International` | 1 si international, 0 sinon | Analyse géographique |
| `Month` | Mois de la commande | Analyse saisonnière |
| `Year` | Année de la commande | Analyse temporelle |
| `Profit_Margin` | Marge bénéficiaire | Analyse financière |

In [11]:
# 4.1. Créer la colonne Delay_Flag (variable cible)
# Late_delivery_risk est déjà 0 ou 1, on va juste la renommer
if 'Late_delivery_risk' in df.columns:
    df['Delay_Flag'] = df['Late_delivery_risk']
    print(" Delay_Flag créée à partir de Late_delivery_risk")

 Delay_Flag créée à partir de Late_delivery_risk


In [12]:
# 4.2. Créer la colonne Delay_Days
# Calcul : Days for shipping (real) - Days for shipment (scheduled)
if 'Days for shipping (real)' in df.columns and 'Days for shipment (scheduled)' in df.columns:
    df['Delay_Days'] = df['Days for shipping (real)'] - df['Days for shipment (scheduled)']
    print(" Delay_Days créée")
    
    # Afficher les statistiques
    print(f" Statistiques de Delay_Days :")
    print(df['Delay_Days'].describe())

 Delay_Days créée
 Statistiques de Delay_Days :
count    180519.000000
mean          0.565807
std           1.490966
min          -2.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           4.000000
Name: Delay_Days, dtype: float64


In [13]:
# 4.3. Créer la colonne Is_International
# Si Order Country n'est pas 'Estados Unidos' (et que les données viennent des Estados Unidos), on considère international
# Adaptez selon le dataset
if 'Order Country' in df.columns:
    # Si le pays n'est pas 'United States', on considère international
    df['Is_International'] = (df['Order Country'] != 'Estados Unidos').astype(int)
    print(" Is_International créée")
    print(f" % International : {df['Is_International'].mean() * 100:.2f}%")

 Is_International créée
 % International : 86.24%


In [14]:
# 4.4. Extraire le mois et l'année
if 'order date (DateOrders)' in df.columns:
    df['Month'] = df['order date (DateOrders)'].dt.month
    df['Year'] = df['order date (DateOrders)'].dt.year
    print(" Month et Year créées")
    
    # Afficher la répartition des mois
    print(" Commandes par mois :")
    print(df['Month'].value_counts().sort_index())

 Month et Year créées
 Commandes par mois :
Month
1     17979
2     14529
3     15919
4     15435
5     15976
6     15139
7     15922
8     15912
9     15489
10    12955
11    12500
12    12764
Name: count, dtype: int64


In [15]:
# 4.5. Créer la colonne Profit_Margin
# Calcul : (Benefit per order / Sales per customer) * 100
if 'Benefit per order' in df.columns and 'Sales per customer' in df.columns:
    # Éviter la division par zéro
    df['Profit_Margin'] = df.apply(
        lambda row: (row['Benefit per order'] / row['Sales per customer']) * 100 
        if row['Sales per customer'] > 0 else 0,
        axis=1
    )
    print(" Profit_Margin créée")
    print(f" Marge bénéficiaire moyenne : {df['Profit_Margin'].mean():.2f}%")

 Profit_Margin créée
 Marge bénéficiaire moyenne : 12.04%


In [16]:
# 4.6. Créer une colonne de saison
def get_season(month):
    if month in [12, 1, 2]:
        return 'Hiver'
    elif month in [3, 4, 5]:
        return 'Printemps'
    elif month in [6, 7, 8]:
        return 'Été'
    else:
        return 'Automne'

if 'Month' in df.columns:
    df['Season'] = df['Month'].apply(get_season)
    print(" Season créée")
    print(df['Season'].value_counts())

 Season créée
Season
Printemps    47330
Été          46973
Hiver        45272
Automne      40944
Name: count, dtype: int64


## 5. Vérification finale et export

In [17]:
# 5.1. Aperçu des données finales
print("Aperçu des données nettoyées :")
df.head()

Aperçu des données nettoyées :


,Order Id,Customer City,Order Country,order date (DateOrders),shipping date (DateOrders),Days for shipping (real),Days for shipment (scheduled),Late_delivery_risk,Delivery Status,Order Item Quantity,Sales per customer,Benefit per order,Category Name,Shipping Mode,Customer Segment,Delay_Flag,Delay_Days,Is_International,Month,Year,Profit_Margin,Season
0,77202,Caguas,Indonesia,2018-01-31 22:56:00,2018-02-03 22:56:00,3,4,0,Advance shipping,1,314.640015,91.250000,Sporting Goods,Standard Class,Consumer,0,-1,1,1,2018,29.001397,Hiver
1,75939,Caguas,India,2018-01-13 12:27:00,2018-01-18 12:27:00,5,4,1,Late delivery,1,311.359985,-249.089996,Sporting Goods,Standard Class,Consumer,1,1,1,1,2018,-80.000645,Hiver
2,75938,San Jose,India,2018-01-13 12:06:00,2018-01-17 12:06:00,4,4,0,Shipping on time,1,309.720001,-247.779999,Sporting Goods,Standard Class,Consumer,0,0,1,1,2018,-80.001291,Hiver
3,75937,Los Angeles,Australia,2018-01-13 11:45:00,2018-01-16 11:45:00,3,4,0,Advance shipping,1,304.809998,22.860001,Sporting Goods,Standard Class,Home Office,0,-1,1,1,2018,7.499754,Hiver
4,75936,Caguas,Australia,2018-01-13 11:24:00,2018-01-15 11:24:00,2,4,0,Advance shipping,1,298.250000,134.210007,Sporting Goods,Standard Class,Corporate,0,-2,1,1,2018,44.999164,Hiver


In [18]:
# 5.2. Statistiques finales
print(f" Dimensions : {df.shape[0]:,} lignes, {df.shape[1]} colonnes")

print("\n Types de données :")
print(df.dtypes.value_counts())

 Dimensions : 180,519 lignes, 22 colonnes

 Types de données :
int64             8
object            7
float64           3
datetime64[ns]    2
int32             2
Name: count, dtype: int64


In [19]:
# 5.3. Vérification des nouvelles colonnes
nouvelles_colonnes = ['Delay_Flag', 'Delay_Days', 'Is_International', 'Month', 'Year', 'Profit_Margin', 'Season']
colonnes_existantes = [col for col in nouvelles_colonnes if col in df.columns]

print(f" {len(colonnes_existantes)} nouvelles colonnes créées sur {len(nouvelles_colonnes)}")
for col in colonnes_existantes:
    print(f"  - {col}: {df[col].nunique()} valeurs uniques")

 7 nouvelles colonnes créées sur 7
  - Delay_Flag: 2 valeurs uniques
  - Delay_Days: 7 valeurs uniques
  - Is_International: 2 valeurs uniques
  - Month: 12 valeurs uniques
  - Year: 4 valeurs uniques
  - Profit_Margin: 60658 valeurs uniques
  - Season: 4 valeurs uniques


In [20]:
# 5.4. Sauvegarde des données propres
df.to_csv('../data/processed/dataco_clean.csv', index=False)
print(" Données propres sauvegardées dans 'data/processed/dataco_clean.csv'")
print(f"   - {df.shape[0]:,} lignes")
print(f"   - {df.shape[1]} colonnes")

 Données propres sauvegardées dans 'data/processed/dataco_clean.csv'
   - 180,519 lignes
   - 22 colonnes


In [21]:
# 5.5. Sauvegarde des colonnes pour Power BI (uniquement les colonnes utiles)
colonnes_powerbi = [
    'Order Id',
    'Customer City',
    'Order Country',
    'order date (DateOrders)',
    'shipping date (DateOrders)',
    'Days for shipping (real)',
    'Days for shipment (scheduled)',
    'Delay_Flag',
    'Delay_Days',
    'Delivery Status',
    'Order Item Quantity',
    'Sales per customer',
    'Benefit per order',
    'Profit_Margin',
    'Category Name',
    'Shipping Mode',
    'Customer Segment',
    'Is_International',
    'Month',
    'Year',
    'Season'
]

# Garder uniquement les colonnes qui existent
colonnes_existantes_powerbi = [col for col in colonnes_powerbi if col in df.columns]
df_powerbi = df[colonnes_existantes_powerbi]

df_powerbi.to_csv('../data/processed/dataco_powerbi.csv', index=False)
print(f" Export Power BI : {len(df_powerbi.columns)} colonnes")

 Export Power BI : 21 colonnes


## 6. Synthèse du jour 2

### Ce que nous avons accompli :

| Tâche | Résultat |
|-------|----------|
| Nettoyage des valeurs manquantes |  Toutes les valeurs manquantes ont été traitées |
| Correction des types de données |  Dates converties en datetime |
| Feature Engineering |  7 nouvelles colonnes créées |
| Export des données |  2 fichiers CSV prêts |

### Données produites :

| Fichier | Description | Utilisation |
|---------|-------------|-------------|
| `dataco_clean.csv` | Données complètes nettoyées | Analyse Python / ML |
| `dataco_powerbi.csv` | Données pour Power BI | Dashboard |

### Prochaines étapes (Jour 3) :
- Importer les données dans Power BI
- Créer les visualisations du tableau de bord
- Définir les KPIs définitifs

---
**Fin du pipeline ETL**

📌 Prochain notebook : `03_ML.ipynb` - Modèle de prédiction des retards
📌 Prochain outil : Power BI - Création du tableau de bord